# Exercise 7.6

## Cross-validation to find optimal degree of polynomial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from ISLP import load_data
#from ISLP.models import (poly, ModelSpec as MS)

In [ ]:
# Load Wage data
Wage = load_data("Wage")
age = Wage["age"]
wage = Wage["wage"]

In [ ]:
# Split intro training/test
np.random.seed(123)
n = len(Wage)
ntrain = n // 2
ind = np.random.choice(n, size=ntrain, replace=False)
train = Wage.iloc[ind]
mask = np.ones(n,dtype=bool)
mask[ind] = False
test = Wage[mask]

In [ ]:
def polyreg(k, data):
    return smf.ols(f"wage ~ np.vander(age, {k+1}, increasing=True)", data=data).fit()

In [ ]:
K = 20
MSE = np.zeros(K)

for k in range(1, K+1):
    reg = polyreg(k,train)
    pred = reg.predict(test)
    MSE[k-1] = np.mean((test["wage"] - pred)**2)

print("MSE for polynomial degrees 1..20:", MSE)
print("Optimal degree:", np.argmin(MSE) + 1)

In [ ]:
# Fit selected polynomials for plotting
reg8 = polyreg(8, Wage)
reg2 = polyreg(2, Wage)

In [ ]:
# Order for plotting
o = np.argsort(Wage["age"])
plt.figure()
plt.scatter(age, wage, color="gray")
plt.plot(Wage["age"].iloc[o], reg8.predict(Wage).iloc[o], color="red", label="Degree 8")
plt.plot(Wage["age"].iloc[o], reg2.predict(Wage).iloc[o], color="blue", label="Degree 2")
plt.xlabel("age")
plt.ylabel("wage")
plt.legend()
plt.show()

## Cross-validation to find optimal cut-off points

In [ ]:
def stepreg(k, data):
    data = data.copy()
    data["age_cut"] = pd.cut(data["age"], k)
    return smf.ols("wage ~ age_cut", data=data).fit()


In [ ]:
MSE_step = np.zeros(K-1)
for k in range(2, K+1):
    reg = stepreg(k, train)
    pred = reg.predict(test.assign(age_cut=pd.cut(test["age"], k)))
    MSE_step[k-2] = np.mean((test["wage"] - pred)**2)

In [ ]:
print("MSE for step functions with 2..20 cuts:", MSE_step)
print("Optimal number of cuts:", np.argmin(MSE_step) + 2)

In [ ]:
# Fit step function with optimal cuts
k = 16
reg = stepreg(k, Wage)
pred = reg.predict(Wage.assign(age_cut=pd.cut(Wage["age"], k)))


In [ ]:
# Plot step function
plt.scatter(age, wage, color="gray")
plt.step(Wage["age"].iloc[o], pred.iloc[o], color="red", where="mid", label="Step k=16")
plt.xlabel("age")
plt.ylabel("wage")
plt.legend()
plt.show()